In [0]:
# %sql
# --Tabla 1: silver_matchs-------------------------------------------------------------------
# SELECT
#     REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
#     name,
#     league,
#     match[0].awayTeam.name AS away_team,
#     match[0].homeTeam.name AS home_team,
#     match[0].location.name AS stadium,
#     cast(match[0].startDate as TIMESTAMP) AS start_date,
#     cast(match[0].endDate as TIMESTAMP) AS end_date,
#     match[0].url AS url
# FROM 
#     futbol.bronze_matchs
# ORDER BY
#     date DESC
# LIMIT 5;

In [0]:
# %sql
# WITH deduplicated AS (
#     SELECT
#         REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
#         name,
#         league,
#         match[0].awayTeam.name AS away_team,
#         match[0].homeTeam.name AS home_team,
#         match[0].location.name AS stadium,
#         cast(match[0].startDate as TIMESTAMP) AS start_date,
#         cast(match[0].endDate as TIMESTAMP) AS end_date,
#         match[0].url AS url,
#         date,
#         ROW_NUMBER() OVER (
#             PARTITION BY 
#                 name,
#                 league,
#                 match[0].awayTeam.name,
#                 match[0].homeTeam.name,
#                 cast(match[0].startDate as TIMESTAMP)
#             ORDER BY date DESC
#         ) AS rn
#     FROM 
#         futbol.bronze_matchs
#     )
#     SELECT
#         match_id,
#         name,
#         league,
#         away_team,
#         home_team,
#         stadium,
#         start_date,
#         end_date,
#         url
#     FROM deduplicated
#     WHERE rn = 1

In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.futbol.silver_matchs;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.futbol.silver_matchs (
    match_id STRING,
    name STRING,
    league STRING,
    away_team STRING,
    home_team STRING,
    stadium STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    url STRING
)
USING DELTA
CLUSTER BY (league, start_date)

In [0]:
%sql
MERGE INTO workspace.futbol.silver_matchs AS target
USING (
    WITH deduplicated AS (
        SELECT
            REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
            name,
            league,
            match[0].awayTeam.name AS away_team,
            match[0].homeTeam.name AS home_team,
            match[0].location.name AS stadium,
            cast(match[0].startDate as TIMESTAMP) AS start_date,
            cast(match[0].endDate as TIMESTAMP) AS end_date,
            match[0].url AS url,
            date,
            ROW_NUMBER() OVER (
                PARTITION BY 
                    name,
                    league,
                    match[0].awayTeam.name,
                    match[0].homeTeam.name,
                    cast(match[0].startDate as TIMESTAMP)
                ORDER BY date DESC
            ) AS rn
        FROM 
            futbol.bronze_matchs
    )
    SELECT
        match_id,
        name,
        league,
        away_team,
        home_team,
        stadium,
        start_date,
        end_date,
        url
    FROM deduplicated
    WHERE rn = 1
) AS source
ON target.name = source.name 
   AND target.league = source.league 
   AND target.away_team = source.away_team
   AND target.home_team = source.home_team
   AND target.start_date = source.start_date
WHEN MATCHED THEN
    UPDATE SET
        *
WHEN NOT MATCHED THEN
    INSERT (
        match_id, 
        name, 
        league, 
        away_team, 
        home_team, 
        stadium, 
        start_date, 
        end_date, 
        url
    ) VALUES (
        source.match_id, 
        source.name, 
        source.league, 
        source.away_team, 
        source.home_team, 
        source.stadium, 
        source.start_date, 
        source.end_date, 
        source.url
    )